<a href="https://www.kaggle.com/code/chuyenhocielts/4-6-matrix-approximation?scriptVersionId=324158963" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# SVD as a Sum of Rank-1 Matrices: The Onion Skin Layering and Image Compression

Historically, we view Singular Value Decomposition as the product of three distinct matrix blocks: $A = U \Sigma V^\top$. However, linear algebra permits us to break this product down into a summation of smaller, simpler matrices—much like peeling the individual layers of an onion:

$$A = \sum_{i=1}^r \sigma_i u_i v_i^\top = \sum_{i=1}^r \sigma_i A_i$$

This formulation reveals a groundbreaking paradigm: **Any complex data matrix or image ($A$) is simply a spatial stacking (summation) of $r$ highly simplified baseline images ($A_i$), each amplified by a distinct structural weight ($\sigma_i$).**

---

## 1. What Exactly is the Baseline Layer $A_i$? (The Rank-1 Grid)

Let's dissect the core component $u_i v_i^\top$ (which we define as the sub-matrix $A_i$):
* $u_i$ is a column vector representing strictly **vertical information**.
* $v_i^\top$ is a row vector representing strictly **horizontal information**.

When you compute the **Outer Product** of a column and a row, it yields a full rectangular matrix. However, this matrix is fundamentally restricted in terms of its information density: **Every column is merely a scaled replica of vector $u_i$, and every row is a scaled replica of vector $v_i^\top$.** 

### Why Rank-1 Layers Look Like Plaid Fabric (Grid Slices)
Because the vertical and horizontal variations are completely independent of one another, **a single Rank-1 matrix cannot form a diagonal line, a circle, or a complex curve.** The only texture a Rank-1 matrix can physically render is the interference of horizontal and vertical color bands. This naturally creates a **plaid, grid-like pattern**. 

To illustrate this with simple numbers, suppose we have:
$$u = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}, \quad v^\top = \begin{bmatrix} 10 & 20 \end{bmatrix}$$

Computing their outer product yields:
$$A_1 = u v^\top = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix} \begin{bmatrix} 10 & 20 \end{bmatrix} = \begin{bmatrix} 10 & 20 \\ 20 & 40 \\ 30 & 60 \end{bmatrix}$$

Notice the rigid pattern: Column 2 is exactly Column 1 multiplied by 2. The row vector $v^\top$ acts purely as a "brightness knob" for each column. It dictates: *"Column 1, print the baseline vertical pattern $u$ at a brightness of 10. Column 2, print that exact same pattern but crank the brightness up to 20."*

Because of this rigid grid structure, the individual early layers of SVD look like coarse plaid grids. To render a complex curve—such as the edge of a rock or a facial feature—the model must stack hundreds of these perpendicular plaid layers on top of each other.

---

## 2. The Singular Value $\sigma_i$ as the Gatekeeper of Dominance

This is where the magic of SVD shines. The singular values $\sigma_i$, sorted in descending order ($\sigma_1 \ge \sigma_2 \ge \dots \ge 0$), act as structural weights determining how much each plaid layer $A_i$ contributes to the final image reconstruction.

* **The First Layer ($A_1$):** Driven by $\sigma_1$ (which possesses an overwhelmingly massive value). This layer captures the absolute core "soul" of the data: it maps out the coarse, global illumination (e.g., the sky region is bright/white, the ground and rock shadows are dark/black). 
* **The Second Layer ($A_2$):** Driven by a significantly smaller $\sigma_2$. This layer begins introducing foundational geometric boundaries, such as rough vertical columns.
* **Subsequent Layers ($A_3, A_4, \dots$):** As $\sigma_i$ plummets toward near-zero values, the corresponding grid layers become ultra-fine and thin. They encode minute, microscopic details: the rough texture of rocks, blades of grass, or minor background artifacts and noise.

---

## 3. The Math of Image Compression: Why Storage Plummets

The realization that we can truncate this summation after $k$ dominant components ($k \ll r$) is the theoretical bedrock of **Truncated SVD** and image compression.



Let's perform a concrete mathematical audit to see exactly how data storage is sliced down:

### A. Traditional Storage Method
Suppose you have a standard grayscale image with a resolution of $1000 \times 1000$ pixels.
* The computer treats the image as a standard dense matrix.
* It must explicitly store every single individual pixel intensity.
* **Total parameters stored:** $1000 \times 1000 = \mathbf{1,000,000} \text{ numbers.}$

### B. Truncated SVD Storage Method
After running SVD, you discover that the first 50 layers ($k = 50$) capture 95% of the vital image details. The remaining 950 layers contain microscopic noise and can be discarded. 

Instead of saving the final reconstructed $1000 \times 1000$ pixel matrix, the computer throws it away and **only saves the raw components required to rebuild the summation formula**:

$$\mathbf{A} \approx \sum_{i=1}^{50} \sigma_i u_i v_i^\top$$

* **Store 50 column vectors $u_i$** (each of length 1000): $50 \times 1000 = 50,000$ numbers.
* **Store 50 row vectors $v_i^\top$** (each of length 1000): $50 \times 1000 = 50,000$ numbers.
* **Store 50 singular weights $\sigma_i$**: 50 numbers.
* **Total parameters stored:** $50,000 + 50,000 + 50 = \mathbf{100,050} \text{ numbers.}$

### The Impact
By shifting from pixel-based storage to concept-based SVD storage, the total number of numbers drops from **1,000,000 to 100,050**. 

You have compressed the file size to **roughly 10% of its original footprint** (a $10\times$ compression ratio). When a user opens the image file, the computer simply initializes a small loop, computes the outer products of the stored vectors, and overlays them on the screen to instantly render the image. The human eye will barely notice any difference!

# Matrix Norms: Decoding the Spectral Norm and the Stretch Factor
$$\|A\|_2 = \max_{x \neq 0} \frac{\|Ax\|_2}{\|x\|_2}$$ 
## 1. Understanding the Ratio $\frac{\|Ax\|_2}{\|x\|_2}$ (The Stretch Factor)

What is the physical, geometric meaning of this ratio? It is simply the **Stretch Factor** of the matrix along the direction of vector $x$.

* **The Intuition:** Imagine you feed an input vector $x$ with a length of $2$ units into the matrix transformation engine $A$. The output vector $Ax$ emerges from the engine with a transformed length of $10$ units. 
* **The Math:** The stretch factor specifically along that chosen path is:
  $$\frac{\|Ax\|_2}{\|x\|_2} = \frac{10}{2} = 5$$

The ratio measures exactly how many times the matrix amplifies or compresses the magnitude of a vector along a given direction.

---

## 2. Why Do We Add the Maximum Constraint ($\max_x$)?

A matrix transformation engine $A$ does not deform space uniformly. It pulls and distorts space unevenly depending on the direction:
* Along one specific direction, it might stretch vectors by a factor of 5.
* Along another perpendicular direction, it might only scale vectors by a factor of 2.
* Along a collapsed dimension (the Null Space), it might crush the vector's length down toward 0 (a stretch factor $< 1$).



The inclusion of the optimization constraint $\max_{x \in \mathbb{R}^n \setminus \{\mathbf{0}\}}$ commands us: 
> *"Test every possible directional arrow across the entire input domain, pass them through the matrix engine $A$, and discover the **absolute maximum stretch factor** that this engine is physically capable of producing!"*

This brings us to the definitive conclusion stated in the textbook: **The Spectral Norm defines the upper bound of spatial distortion—it measures the maximum possible factor by which a vector $x$ can be elongated when pre-multiplied by $A$.** This maximum capability is the exact definition of the Spectral Norm $\|A\|_2$.


# Determinants vs. Spectral Norm: Global Volume vs. Local Stretching

To truly master linear algebra, one must distinguish between the "Global" perspective (Determinants) and the "Local" or "Directional" perspective (SVD/Spectral Norm).

---

## 1. The Global Perspective: The Determinant ($\det(A)$)
The determinant is a **Global** measurement. It does not care about the individual orientation or distortion of any specific vector; it only measures the "net change in mass" of space.

* **Geometric Intuition:** Think of the determinant as the **"Volume Scaling Factor"** of the transformation.
* **The Math:** If the unit square (area = 1) is transformed by matrix $A$ into a parallelogram with area 6, then $\det(A) = 6$.
* **The Blind Spot:** Because it aggregates everything into a single product, the determinant cannot tell you if the space was stretched uniformly or if it was pulled thin in one direction while compressed in another. It only reports the final "total volume" occupied by the space after the transformation.

---

## 2. The Local Perspective: Spectral Norm and SVD ($\sigma_i$)
The Spectral Norm ($\|A\|_2$) and the Singular Values ($\sigma_i$) provide a **Local** (or Directional) breakdown of how space is deformed. 

* **Geometric Intuition:** Think of matrix $A$ as a pair of hands grabbing a sheet of rubber. You might pull the rubber hard to the left (stretching the horizontal axis) while pushing down from the top (compressing the vertical axis). 
* **The Math:** The Spectral Norm ($\|A\|_2$) and the individual singular values ($\sigma_i$) measure these specific, directional deformations. 
    * One direction might stretch space by a factor of 5 ($\sigma_1$).
    * Another direction might compress space by a factor of 0.5 ($\sigma_2$).
* **The "Radar" Intuition:** The equation $\|A\|_2 = \max_{x \neq 0} \frac{\|Ax\|_2}{\|x\|_2}$ is like standing at the origin and rotating a "radar stick" (vector $x$) $360^\circ$ around the space. At every angle, the system measures the length of the output vector vs. the input vector. The maximum value that radar stick ever hits is your Spectral Norm ($\sigma_1$).



---

## 3. The Unification: Bridging Volume and Length
There is a beautiful, elegant bridge that connects your memory of "Volume" (Determinant) to the "Length" of individual axes (SVD). 

If the SVD transformation turns a sphere into an ellipsoid, the "total volume" of that ellipsoid is simply the product of all its principal axes. Therefore, the magnitude of the determinant is exactly equal to the product of all singular values:

$$\boxed{|\det(A)| = \sigma_1 \times \sigma_2 \times \dots \times \sigma_n}$$

---

## Summary Comparison Table

| Feature | Determinant ($\det(A)$) | Spectral Norm / SVD ($\sigma_i$) |
| :--- | :--- | :--- |
| **Perspective** | **Global** (Aggregate total). | **Local** (Directional focus). |
| **Physical Meaning** | How much the **total volume** of space expanded or contracted. | How much space is stretched **along specific directions**. |
| **Sensitivity** | Sensitive to orientation (sign can be $\pm$). | Sensitive to the most extreme distortion ($\sigma_{\max}$). |
| **Usage Case** | Checking if a matrix is invertible (is volume 0?). | Measuring noise, data variance, and compression power. |

**The Final Takeaway:** * If you want to know if the entire space has "collapsed" (singular matrix) $\rightarrow$ **Look at the Determinant.**
* If you want to know the "maximum power" of the matrix to deform space (or identify the strongest signals in your data) $\rightarrow$ **Look at the Spectral Norm ($\sigma_1$) and the SVD axes.**

# Singular Values ($\sigma$) vs. Eigenvalues ($\lambda$): Distinguishing Physical Stretch from Spatial Orientation

A common pitfall in linear algebra is conflating singular values ($\sigma$) with eigenvalues ($\lambda$). While they are deeply linked, they occupy different conceptual territories.

---

## 1. The Disconnect: Why $\sigma_i \neq \lambda_i$ for General Matrices

For any matrix $A$ (square or rectangular), the definition of a singular value is **unique**:
$$\sigma_i = \sqrt{\lambda_i(A^\top A)}$$

**The Misconception:** Many assume that for square matrices, $\sigma_i$ simply equals $\lambda_i(A)$. This is categorically false for most matrices.

* **Example:** Let $A = \begin{bmatrix} -5 & 0 \\ 0 & -5 \end{bmatrix}$. 
    * **The Eigenvalue ($\lambda$):** $-5$. (Eigenvalues can be negative or even complex numbers, as they track directional flipping).
    * **The Singular Value ($\sigma$):** $5$. (Singular values must be non-negative, as they represent the physical "stretch length").
* **The Rule:** Only when a matrix is **Symmetric and Positive Semi-Definite** (all $\lambda_i \ge 0$) do the eigenvalues coincide with the singular values. In all other cases, they remain distinct mathematical entities.

---

## 2. The Bridge: Deriving $|\det(A)| = \prod \sigma_i$

The formula linking the determinant to singular values is a perfect demonstration of the SVD structure in action. Assume $A$ is a square $n \times n$ matrix:

1. **Decomposition:** Start with $A = U \Sigma V^\top$.
2. **Determinant Property:** Apply the multiplicative property $\det(XYZ) = \det(X)\det(Y)\det(Z)$:
   $$\det(A) = \det(U) \cdot \det(\Sigma) \cdot \det(V^\top)$$
3. **The Orthogonal Advantage:** Because $U$ and $V$ are orthogonal matrices, their determinants are constrained to $\pm 1$:
   $$\det(A) = (\pm 1) \cdot (\sigma_1 \times \sigma_2 \times \dots \times \sigma_n) \cdot (\pm 1)$$
4. **Conclusion:** Taking the absolute value eliminates the directional flipping artifacts:
   $$\boxed{|\det(A)| = \sigma_1 \times \sigma_2 \times \dots \times \sigma_n}$$

---

## 3. The Grand Unification: Eigenvalues vs. Singular Values

We know from standard linear algebra that the determinant of a square matrix is also the product of its eigenvalues:
$$\det(A) = \lambda_1 \times \lambda_2 \times \dots \times \lambda_n$$

By combining our two findings, we arrive at a profound geometric truth:
$$\boxed{|\lambda_1 \times \lambda_2 \times \dots \times \lambda_n| = \sigma_1 \times \sigma_2 \times \dots \times \sigma_n}$$



### The Philosophical Takeaway

| Perspective | Role in Geometric Transformation | Mathematical Meaning |
| :--- | :--- | :--- |
| **Eigenvalues ($\lambda$)** | Captures the **"Spatial Orientation"** of the transformation. | Tracks how space is rotated, flipped, and scaled. The sign ($\pm$) tells you if the orientation was inverted. |
| **Singular Values ($\sigma$)** | Captures the **"Physical Magnitude"** of the stretching. | Tracks the pure physical "stretch length" of the principal axes. It completely ignores spatial flipping, which is why SVD is strictly non-negative. |

**Final Intuition:**
* The **Eigen-world** cares about the "integrity" of the transformation—did space flip? Did it rotate? 
* The **SVD-world** cares about the "energy" of the transformation—how much did the machine stretch the space, regardless of which way the vectors are pointing? 

This is why SVD is the "gold standard" for Machine Learning compression and dimensionality reduction: it isolates the physical strength of the data signals ($\sigma$) from the confusing directional artifacts of the specific coordinate system.

# Mathematical Proof: The Spectral Norm is the Largest Singular Value ($\|A\|_2 = \sigma_1$)

We aim to prove that the maximum stretch factor of a matrix $A$ is precisely its largest singular value ($\sigma_1$):
$$\|A\|_2 = \max_{x \neq 0} \frac{\|Ax\|_2}{\|x\|_2} = \sigma_1$$

---

## Step 1: Normalizing the Input Space
Instead of analyzing vectors $x$ of arbitrary length, we constrain the search to unit vectors $v$ with $\|v\|_2 = 1$. This is valid because we can always define $v = \frac{x}{\|x\|_2}$. The denominator disappears, simplifying the optimization problem to:
$$\text{Maximize } \|Av\|_2 \quad \text{subject to } \|v\|_2 = 1$$

## Step 2: Squaring the Goal (The $A^\top A$ Catalyst)
Working with the square root of the Euclidean norm is cumbersome. We instead maximize the squared length:
$$\|Av\|_2^2 = (Av)^\top (Av) = v^\top A^\top A v$$
The appearance of the symmetric matrix $A^\top A$ allows us to leverage the **Spectral Theorem**.

## Step 3: Decomposing into the Eigen-World
Since $A^\top A$ is symmetric, it possesses an orthonormal basis of eigenvectors $\{v_1, v_2, \dots, v_n\}$ with corresponding eigenvalues $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_n \ge 0$. 

Any unit vector $v$ can be represented as a linear combination of these basis vectors:
$$v = c_1 v_1 + c_2 v_2 + \dots + c_n v_n$$
By the Pythagorean theorem in $n$ dimensions, the unit constraint implies:
$$\sum_{i=1}^n c_i^2 = 1$$

## Step 4: The Budget Allocation Game
Substituting our decomposed vector $v$ into the squared norm expression yields:
$$v^\top A^\top A v = v^\top A^\top A \left( \sum_{i=1}^n c_i v_i \right) = v^\top \left( \sum_{i=1}^n c_i \lambda_i v_i \right)$$

Because the eigenvectors are orthonormal ($v_i^\top v_j = 0$ for $i \neq j$ and $v_i^\top v_i = 1$), the cross-terms vanish, leaving:
$$\|Av\|_2^2 = \sum_{i=1}^n \lambda_i c_i^2 = \lambda_1 c_1^2 + \lambda_2 c_2^2 + \dots + \lambda_n c_n^2$$



**The Financial Intuition:**
* You have a total investment budget of $1$ (since $\sum c_i^2 = 1$).
* You have $n$ investment channels with returns $\lambda_1, \lambda_2, \dots, \lambda_n$.
* To maximize your total return ($\|Av\|_2^2$), you must allocate your entire budget to the channel with the highest return, $\lambda_1$.

Thus, to achieve the maximum, we set $c_1^2 = 1$ and $c_2 = \dots = c_n = 0$.

## Step 5: Final Conclusion
The maximum value of the squared norm is $\lambda_1$. Taking the square root to retrieve the original length:
$$\max \|Av\|_2 = \sqrt{\lambda_1} = \sigma_1$$

**Geometric Q.E.D.:** The maximum stretch factor $\|A\|_2$ is exactly $\sigma_1$. To achieve this maximum, your input vector $v$ must align perfectly with the first right-singular vector $v_1$. The matrix $A$ will then stretch this vector along the direction of the first left-singular vector $u_1$ by the factor $\sigma_1$.